# 📚 Reader3 Pro - Book Data Exploration

探索 book.pkl 文件的数据结构，并实现一些有趣的功能：
- 查看书籍元数据和结构
- 内容统计分析
- 使用 RAG 进行语义搜索
- 章节摘要生成

**示例书籍**: The Cold Start Problem

## 1. 加载书籍数据

In [ ]:
import pickle
import os
from pathlib import Path
from typing import List, Dict
from dataclasses import asdict
import json

# 加载 book.pkl
book_path = "../The-Cold-Start-Problem_data/book.pkl"

print(f"📖 Loading book from: {book_path}")
with open(book_path, 'rb') as f:
    book = pickle.load(f)

print(f"✅ Book loaded successfully!")
print(f"Type: {type(book).__name__}")

## 2. 探索书籍元数据

In [ ]:
# 查看书籍元数据
print("📊 Book Metadata:")
print("=" * 50)
print(f"Title: {book.metadata.title}")
print(f"Authors: {', '.join(book.metadata.authors)}")
print(f"Publisher: {book.metadata.publisher}")
print(f"Language: {book.metadata.language}")
print(f"Identifiers: {book.metadata.identifiers}")
print()
print(f"📄 Total Chapters (Spine): {len(book.spine)}")
print(f"🗂️ TOC Entries: {len(book.toc)}")
print(f"🖼️ Images: {len(book.images)}")

## 3. 章节结构分析

In [ ]:
# 分析章节
print("📑 Chapter Analysis:")
print("=" * 50)

total_text_length = 0
chapter_stats = []

for i, chapter in enumerate(book.spine):
    text_len = len(chapter.text)
    total_text_length += text_len
    word_count = len(chapter.text.split())
    
    chapter_stats.append({
        'order': chapter.order,
        'title': chapter.title,
        'chars': text_len,
        'words': word_count
    })

# 显示前 10 章
print(f"\nFirst 10 Chapters:")
for stat in chapter_stats[:10]:
    print(f"  [{stat['order']:2d}] {stat['title'][:50]:50s} | {stat['words']:,} words")

print(f"\n📊 Statistics:")
print(f"  Total text length: {total_text_length:,} characters")
print(f"  Total words: {sum(s['words'] for s in chapter_stats):,}")
print(f"  Average chapter length: {total_text_length // len(book.spine):,} characters")

## 4. 内容预览

In [ ]:
# 查看第一章内容
first_chapter = book.spine[0]

print("📖 First Chapter Preview:")
print("=" * 50)
print(f"Title: {first_chapter.title}")
print(f"ID: {first_chapter.id}")
print(f"\nFirst 500 characters:")
print("-" * 50)
print(first_chapter.text[:500])
print("...")

## 5. 简单的关键词搜索

In [ ]:
import re

def search_keyword(book, keyword: str, case_sensitive: bool = False) -> List[Dict]:
    """在书籍中搜索关键词"""
    results = []
    
    flags = 0 if case_sensitive else re.IGNORECASE
    pattern = re.compile(re.escape(keyword), flags)
    
    for chapter in book.spine:
        matches = list(pattern.finditer(chapter.text))
        
        for i, match in enumerate(matches):
            start = match.start()
            end = match.end()
            
            # 提取上下文
            context_start = max(0, start - 50)
            context_end = min(len(chapter.text), end + 50)
            
            context = chapter.text[context_start:context_end]
            
            results.append({
                'chapter': chapter.title,
                'chapter_order': chapter.order,
                'occurrence': i,
                'position': start,
                'context': context,
                'match': chapter.text[start:end]
            })
    
    return results

# 测试搜索
search_term = "network"
print(f"🔍 Searching for: '{search_term}'")
print("=" * 50)

results = search_keyword(book, search_term)
print(f"\nFound {len(results)} occurrences\n")

# 显示前 5 个结果
for i, result in enumerate(results[:5], 1):
    print(f"[{i}] Chapter {result['chapter_order']}: {result['chapter']}")
    print(f"    Context: ...{result['context']}...")
    print()

## 6. RAG 语义搜索

使用 ChromaDB 进行语义搜索，找到与查询相关的章节

In [ ]:
# 检查是否安装了 ChromaDB
try:
    import chromadb
    from chromadb.utils import embedding_functions
    
    print("✅ ChromaDB is available!")
    print("Now we can create embeddings and perform semantic search.")
    
except ImportError:
    print("⚠️ ChromaDB not installed. Run: uv add chromadb sentence-transformers")
    print("Skipping RAG section...")

In [ ]:
# 创建临时的 ChromaDB 集合用于演示
import chromadb
from chromadb.utils import embedding_functions

# 初始化 ChromaDB 客户端
client = chromadb.Client()

# 使用多语言嵌入模型
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

# 创建集合
print("📊 Creating vector collection...")
collection = client.create_collection(
    name="book_chapters",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

print("✅ Collection created!")

In [ ]:
# 将章节内容添加到向量数据库（这可能需要一些时间）
print("🔄 Indexing chapters... (this may take a minute)")

documents = []
metadatas = []
ids = []

for chapter in book.spine:
    # 对于长章节，我们可以分块
    # 这里为了演示，我们使用每章的前 2000 个字符
    text_chunk = chapter.text[:2000] if len(chapter.text) > 2000 else chapter.text
    
    chunk_id = f"chapter_{chapter.order}"
    
    documents.append(text_chunk)
    metadatas.append({
        "chapter_id": chapter.id,
        "chapter_title": chapter.title,
        "chapter_order": chapter.order,
        "full_length": len(chapter.text)
    })
    ids.append(chunk_id)

# 批量添加到集合
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"✅ Indexed {len(documents)} chapters!")

In [ ]:
# 执行语义搜索
def semantic_search(query: str, n_results: int = 5):
    """使用 RAG 进行语义搜索"""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )
    
    return results

# 测试语义搜索
query = "How do companies solve the cold start problem?"
print(f"🔍 Semantic Search Query: '{query}'")
print("=" * 50)

results = semantic_search(query, n_results=5)

print(f"\nTop 5 Most Relevant Chapters:\n")
for i in range(len(results['ids'][0])):
    metadata = results['metadatas'][0][i]
    distance = results['distances'][0][i]
    similarity = 1 - distance  # Convert distance to similarity
    
    print(f"[{i+1}] Chapter {metadata['chapter_order']}: {metadata['chapter_title']}")
    print(f"    Similarity: {similarity:.3f}")
    print(f"    Preview: {results['documents'][0][i][:200]}...")
    print()

## 7. 内容分析与可视化

In [ ]:
# 词频统计
from collections import Counter
import re

def get_word_frequency(book, top_n: int = 20):
    """统计全书词频"""
    all_text = " ".join([chapter.text for chapter in book.spine])
    
    # 简单的词语提取（可以改进）
    words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text.lower())
    
    # 过滤常见停用词
    stop_words = {'the', 'and', 'for', 'that', 'this', 'with', 'was', 'are', 
                  'have', 'from', 'not', 'but', 'they', 'you', 'can', 'will'}
    
    words = [w for w in words if w not in stop_words]
    
    return Counter(words).most_common(top_n)

# 获取最常见的词
print("📊 Top 20 Most Frequent Words:")
print("=" * 50)

word_freq = get_word_frequency(book, top_n=20)

for word, count in word_freq:
    bar = '█' * (count // 100)  # 简单的条形图
    print(f"{word:15s} {count:5d} {bar}")

## 8. 实用功能：章节摘要

In [ ]:
def extract_chapter_sentences(chapter_text: str, num_sentences: int = 3) -> str:
    """提取章节的关键句子（简单的启发式方法）"""
    # 分句
    sentences = re.split(r'[.!?]+', chapter_text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    
    if len(sentences) <= num_sentences:
        return " ".join(sentences)
    
    # 简单策略：取开头、中间、结尾的句子
    selected = [
        sentences[0],  # 开头
        sentences[len(sentences)//2],  # 中间
        sentences[-1]  # 结尾
    ]
    
    return " [...] ".join(selected)

# 为前几章生成摘要
print("📝 Chapter Summaries:")
print("=" * 50)

for i in range(min(5, len(book.spine))):
    chapter = book.spine[i]
    summary = extract_chapter_sentences(chapter.text, num_sentences=3)
    
    print(f"\n[Chapter {chapter.order}] {chapter.title}")
    print(f"Summary: {summary[:300]}...")

## 9. 保存分析结果

In [ ]:
# 导出书籍元数据到 JSON
book_info = {
    'title': book.metadata.title,
    'authors': book.metadata.authors,
    'publisher': book.metadata.publisher,
    'language': book.metadata.language,
    'chapters': [
        {
            'order': ch.order,
            'title': ch.title,
            'word_count': len(ch.text.split()),
            'char_count': len(ch.text)
        }
        for ch in book.spine
    ],
    'statistics': {
        'total_chapters': len(book.spine),
        'total_words': sum(len(ch.text.split()) for ch in book.spine),
        'total_characters': sum(len(ch.text) for ch in book.spine),
        'total_images': len(book.images)
    }
}

# 保存到文件
output_file = "book_analysis.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(book_info, f, indent=2, ensure_ascii=False)

print(f"✅ Analysis saved to: {output_file}")
print(f"\n📊 Summary:")
print(f"  Total chapters: {book_info['statistics']['total_chapters']}")
print(f"  Total words: {book_info['statistics']['total_words']:,}")
print(f"  Total characters: {book_info['statistics']['total_characters']:,}")

## 10. 实验区：你的自定义代码

在这里添加你自己的探索代码！

In [ ]:
# 你的实验代码
# 例如：
# - 比较不同章节的文本相似度
# - 提取关键词和主题
# - 使用 AI API 生成章节摘要
# - 创建知识图谱
# - 分析写作风格

print("🚀 Start your experiments here!")